In [1]:
import os
from dotenv import load_dotenv

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

load_dotenv()

True

In [54]:
cemb = pd.read_parquet('clusters_representatives_with_embeddings_Qwen3-8B.parquet')
cemb.head()

,cluster_id,text,count,embedding
0,0,Insurance programs and tax incentives for floo...,715,"[0.013568851165473461, 0.02100222185254097, -0..."
1,1,Preserving natural lands for flood management,447,"[0.02139049768447876, -0.0014260332100093365, ..."
2,2,Continuous monitoring of fishways to optimise ...,1536,"[-0.0037227999418973923, -0.009593934752047062..."
3,3,Increasing restoration funding,79,"[-0.011206850409507751, 0.02045968547463417, -..."
4,4,Implementation of measures to mitigate the ris...,1031,"[0.009541078470647335, 0.029512155801057816, -..."


In [5]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='library-v1'), CollectionDescription(name='policies-v1')])

In [6]:
collection = "clusters-v1"
dim = 4096

In [7]:
client.create_collection(
    collection_name=collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

True

In [8]:
imp = pd.read_parquet('../results_557k/cluster_impacts_2026-03-12.parquet')
imp.head()

,cluster_id,impact_category,impact_dim,negative,neutral,positive,negative_refs,neutral_refs,positive_refs
0,0,Human Needs,Shelter and living conditions,0,0,13,[],[],"[W2003373721_1, W2156345725_0, W2289768102_1, ..."
1,0,Justice,Distributional,0,2,0,[],"[W2557096622_1, W4389432677_0]",[]
2,0,Planetary Boundaries,Climate change,0,0,1,[],[],[W4415811729_2]
3,0,Wellbeing,Community,0,0,1,[],[],[W4287095827_3]
4,0,Wellbeing,Environment,0,0,8,[],[],"[W1000416519_1, W2156345725_0, W2557096622_1, ..."


In [28]:
d = {}
for (cid, cat, impact), group in imp.groupby(['cluster_id', 'impact_category', 'impact_dim']):
    tmp = group.drop(['cluster_id', 'impact_category', 'impact_dim'], axis=1)
    tmp = tmp.map(lambda x: x.tolist() if hasattr(x, 'tolist') else x)
    payload = tmp.to_dict(orient='records')
    if cid not in d:
        d[cid] = {cat: {impact: payload}}
    else:
        if cat not in d[cid]:
            d[cid][cat] = {impact: payload}
        else:
            d[cid][cat][impact] = payload

In [57]:
cemb['impacts'] = cemb['cluster_id'].map(lambda x: d.get(x, {}))

In [59]:
vectors = cemb.pop('embedding')
payloads = [row.to_dict() for _, row in cemb.iterrows()]

In [62]:
client.upload_collection(
    collection,
    vectors=vectors,
    payload=payloads,
    parallel=4,
    batch_size=100
)

In [63]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='library-v1'), CollectionDescription(name='clusters-v1'), CollectionDescription(name='policies-v1')])